# Prep

## Import stuff

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math

## Some magical magic to make the R stuff work

In [2]:
import os
os.environ["OMP_NUM_THREADS"], os.environ["OPENBLAS_NUM_THREADS"], os.environ["MKL_NUM_THREADS"] 

('1', '1', '1')

In [3]:
# fix from here https://github.com/rpy2/rpy2/issues/882
import os
#os.environ['R_HOME'] = '/Users/zeleninam2/miniconda3/envs/env_hitop_3way_2026/lib/R' # change this to whatever is relevant to you, or you might not need this line at all.

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

import rpy2.ipython.html
rpy2.ipython.html.init_printing()
from rpy2.rinterface_lib.embedded import RRuntimeError

rbase = importr('base')
utils = importr('utils')
lavaan = importr('lavaan')
semtools = importr('semTools')

# Paths

In [4]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'

## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [5]:
total_cpus = os.cpu_count()
cpus_to_use = total_cpus-2
global cpus_to_use
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000
global num_iter


Going to use 14 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [6]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [7]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x144a377d0> [0]

### TEST THE SEEDS!!!!!!!!

In [8]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [9]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [18]:
def load_item_lookup(measure='HiTOP'):
    """
    possible values for measure='BAARS-IV','GAD-7','PHQ-8','HiTOP',
    """
    # this is a table that aligns scale items with their corresponding questions
    htqs = pd.read_excel(path_to_item_lookup, skiprows=2,
                                names=['id', 'scale', 'item', '_0', '_1', '_2', '_3']).drop(['_0', '_1', '_2', '_3'], axis=1)
    # cmqs is a dataframe with items and responses
    # htcm is the same dataframe, but only displaying HiTOP items
    cmqs = pd.read_csv(path_to_cogmood_questions)
    cmqs = cmqs.rename({'Unnamed: 0': 'id'}, axis=1)
    htcm = cmqs.query("measure == @measure")
    if measure == 'BAARS-IV':
        return htcm
    elif measure != 'HiTOP':
        m_string = measure.lower().split('-')[0]
        lut = phq_lut = {f'{m_string}_{ix +1}': tt for ix, tt in enumerate(htcm.item.values)}
        return lut
    # this is a lookup table for subscales, items (sentence questions), and hitop ids
    item_lookup = htcm.loc[:, ['id','subscale', 'item']].merge(htqs.loc[:, ['id','item']], how='inner', on='item', suffixes=['_cm', '_ht'])
    item_lookup['htid'] = 'hitop'+ item_lookup.id_ht.astype(str)
    return(item_lookup)
item_lookup = load_item_lookup('HiTOP')
item_lut = {row.htid: row.item for row in item_lookup.loc[:, ['item', 'htid']].itertuples()}
phq_lut = load_item_lookup('PHQ-8')
gad_lut = load_item_lookup('GAD-7')
baars_lookup = load_item_lookup('BAARS-IV')
baars_lut = {}
for subscale in ['inattention', 'sct', 'hyperactivity', 'impulsivity']:
    if subscale == 'sct':
        ss_items = baars_lookup.loc[baars_lookup.subscale == 'Sluggish Cognitive Tempo', 'item'].values
    else:
        ss_items = baars_lookup.loc[baars_lookup.subscale == subscale.title(), 'item'].values
    baars_lut[subscale] = {f'{subscale}_{ix+1}': tt for ix, tt in enumerate(ss_items)}
def print_summary(obj):
    return rprint(summary(obj))

def print_short_summary(invariance_output):
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    ro.r("stringid <- grep(\"chisq\", myoutput)")
    ro.r("print(myoutput[stringid-1])")
    ro.r("print(myoutput[stringid])")
    return (0)

def print_problematic_items(invariance_output):
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    print('Problematic items:')
    ro.r("stringidsig <- grep(\"may differ between Groups\", myoutput)")
    ro.r("print(myoutput[stringidsig])")   
    return (0)
    
def extract_p(invariance_output):
    ro.globalenv['invariance_output'] = invariance_output
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    ro.r("stringid <- grep(\"chisq\", myoutput)")
    ro.r('myrline <- myoutput[stringid]')
    mypline = ro.r('myrline')[0]
    my_p = mypline.split()[-1]
    return(my_p)

def nafloat(x):
    try:
        x = float(x)
    except ValueError:
        x = np.nan
    return x
        

def check_secondary_criteria(fit_config):
    Criteria_passed = False
    ro.r("myoutputconfig <- capture.output(summary(fit_config, fit.measures=TRUE))")
    ro.r("stringid_cfi <- grep(\"Robust.*CFI\", myoutputconfig)")
    ro.r("cfi_line <- myoutputconfig[stringid_cfi]")
    cfiline = ro.r('cfi_line')[0]    
    my_cfi = cfiline.split()[-1] 
    my_cfi = nafloat(my_cfi)
    ro.r("stringid_tli <- grep(\"Robust.*TLI\", myoutputconfig)")
    ro.r("tli_line <- myoutputconfig[stringid_tli]")
    tliline = ro.r('tli_line')[0]    
    my_tli = tliline.split()[-1] 
    my_tli = nafloat(my_tli)
    ro.r("stringid_rmsea <- grep(\"Robust.*RMSEA\", myoutputconfig)")
    ro.r("rmsea_line <- myoutputconfig[stringid_rmsea]")
    myrmsealine = ro.r('rmsea_line')[0]
    my_rmsea = myrmsealine.split('\n')[0].split()[-1]
    my_rmsea = nafloat(my_rmsea)   
    if (my_cfi > 0.95) and (my_tli > 0.95) and (my_rmsea < 0.06):
        Criteria_passed = True
    return (Criteria_passed, my_cfi, my_tli, my_rmsea)   

def check_hitop_ids(items_to_check, my_item_lookup):
    # checks what item ids mean what
    for i in items_to_check:
        a = repr(my_item_lookup.loc[my_item_lookup['htid'] == 'hitop'+i]['htid']).split(' ', 1)[1]
        aa = str(a).split('\n')[0]
        b = repr(my_item_lookup.loc[my_item_lookup['htid'] == 'hitop'+i]['item']).split(' ', 1)[1]
        bb = str(b).split('\n')[0]
        print(aa+bb)
    print('\n')

def cfa_helper_func(scalename, list_of_items, do_metric, do_scalar, do_strict, mydata_python, mydata_temp_path):
    # Helper CFA funct
    # Inputs:
    # --> scalename - string of scale name: 'appetite_loss'
    # --> list_of_items - list of items to test, not necessarily the full list of items for the scale: ['hitop280', 'hitop283', 'hitop109']
    # --> do_metric, do_scalar, do_strict: True/False
    # --> mydata_python - python df with data (this function will feed it to R)
    # --> mydata_temp_path - path to do an intermediate step of saving from python, then reading into R

    # SETTING THE SEED HERE, JUST TO BE SURE IT IS SET TO THE SAME THING EVERY TIME I RUN THIS HELPER FUNCTION
    random.seed(12345)
    ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
    ro.r('set.seed(12345)')

    rprint = ro.r('print')
    summary = ro.r('summary')

    # make a R-readable string out of scalename and desired list of items
    # (this will sometimes match the full set - and it's okay - but sometimes won't, when we iterate through all options looking for inv subsets)
    myrstring = scalename + "=~"
    for x in list_of_items: # iterate through desired items and add them to my formula
        myrstring += x + ' + '
    myrstring = myrstring[:-3] # drop the last " + "
    #print(f'Testing formula: {myrstring}')      
    ro.globalenv['testformula_r'] = myrstring # send it to the R world

    # transfer data from python into R
    # I save the python df as csv and then load it to R (there must be a more elegant way to do this, but this one works...)
    mydata_python.to_csv(mydata_temp_path)
    r_data_command = f'rdata <- read.csv(\'{mydata_temp_path}\', header=TRUE)'
    ro.r(r_data_command)

    # telling R which column to test
    group = 'whichdata'
    ro.globalenv['group'] = group

    flag_passed_metric = False # setting flag for metric because this level is the most important one --> we return this

    # Build R character vector of item names Suggestion from Claude for specifying ordinal items
    ordered_str = 'c(' + ', '.join(f'"{i}"' for i in list_of_items) + ')'
       
    # Always do CONFIGURAL:
    flag_passed_config = False # flag for configural
    fit_config = ro.r(f'cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", parameterization = "theta", ordered = {ordered_str})')
    ro.globalenv['fit_config'] = fit_config        
    out_config = semtools.permuteMeasEq(nPermute=num_iter,
                                        con=fit_config, # In the case of testing configural invariance when modelType = "mgcfa", con is the configural model (implicitly, the unconstrained model is the saturated model, so use the defaults uncon = NULL and param = NULL).
                                        parallelType="multicore", ncpus=cpus_to_use)
    config_p = extract_p(out_config)
    if float(config_p) >= 0.05:
        print('CONFIG INVARIANT chisq p = ' + str(config_p))
        flag_passed_config = True
    else:
        (flag_passed_config, my_cfi, my_tli, my_rmsea) = check_secondary_criteria(fit_config)
        if flag_passed_config:
            print('CONFIG chisq p = ' + str(config_p))
            print('PASSED SECONDARY CRITERIA WITH CFI = ' + str(my_cfi) + ' TLI = ' + str(my_tli) + ' RMSEA = ' + str(my_rmsea))
        else:
            print('CONFIG chisq p = ' + str(config_p))
            print('FAILED SECONDARY CRITERIA WITH CFI = ' + str(my_cfi) + ' TLI = ' + str(my_tli) + ' RMSEA = ' + str(my_rmsea))

    if flag_passed_config: # only do metric if configural is passed
        if do_metric:
            # ====== METRIC =====
            fit_metric = ro.r(f'cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", parameterization = "theta", group.equal="loadings",  ordered = {ordered_str})')
            ro.globalenv['fit_metric'] = fit_metric
            out_metric = semtools.permuteMeasEq(nPermute=num_iter, 
                                                        uncon=fit_config, 
                                                        con=fit_metric, 
                                                        param="loadings",
                                                        parallelType="multicore", ncpus=cpus_to_use)
            metric_p = extract_p(out_metric)
            if float(metric_p) >= 0.05:
                print('METRIC INVARIANT chisq p = ' + str(metric_p))
                flag_passed_metric = True

                if do_scalar: # only do scalar if metric is passed
                    # ====== SCALAR =====
                    fit_scalar = ro.r(f'cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", parameterization = "theta", group.equal=c("loadings", "intercepts"),  ordered = {ordered_str})')
                    ro.globalenv['fit_scalar'] = fit_scalar
                    out_scalar = semtools.permuteMeasEq(nPermute=num_iter, 
                                                            uncon=fit_metric, 
                                                            con=fit_scalar, 
                                                            param=ro.StrVector(["loadings","intercepts"]),
                                                            parallelType="multicore", 
                                                            ncpus=cpus_to_use)
                    scalar_p = extract_p(out_scalar)
                    if float(scalar_p) >= 0.05:
                        print('SCALAR INVARIANT chisq p = ' + str(scalar_p)) 
                        
                        if do_strict: # only do strict if scalar is passed
                            # ====== STRICT =====
                            ro.globalenv['out_scalar'] = out_scalar
                            fit_strict = ro.r(f'cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", parameterization = "theta", group.equal=c("loadings", "intercepts", "residuals"),  ordered = {ordered_str})')
                            ro.globalenv['fit_strict'] = fit_strict
                            out_strict = semtools.permuteMeasEq(nPermute=num_iter, 
                                                                        uncon=fit_scalar, 
                                                                        con=fit_strict, 
                                                                        param=ro.StrVector(["loadings","intercepts", "residuals"]),
                                                                        parallelType="multicore", 
                                                                        ncpus=cpus_to_use)
                            strict_p = extract_p(out_strict)
                            if float(strict_p) >= 0.05:
                                print('STRICT INVARIANT chisq p = ' + str(strict_p))
                            else:
                                # significant:
                                print('STRICT INVARIANT NOT PASSED, chisq p = ' + str(strict_p))
                        else: # not do_strict
                            print('STRICT N/A')
                            strict_p = 'NA'
                    else: # scalar not passed
                        print('SCALAR INVARIANT NOT PASSED, chisq p = ' + str(scalar_p))
                        print('STRICT N/A')
                        strict_p = 'NA' 
                else: # not do_scalar
                    print('SCALAR N/A')
                    scalar_p = 'NA' 
                    print('STRICT N/A')
                    strict_p = 'NA'                     
            else: # metric not passed
                print('METRIC INVARIANT NOT PASSED, chisq p = ' + str(metric_p)) 
                print('SCALAR N/A')
                print('STRICT N/A')                    
                scalar_p = 'NA'
                strict_p = 'NA'       
        else: # not do_metric
            print('METRIC N/A')
            print('SCALAR N/A')
            print('STRICT N/A')
            metric_p = 'NA'
            scalar_p = 'NA'
            strict_p = 'NA'            
    else: # config not passed
        print('CONFIG INVARIANT NOT PASSED, chisq p = ' + str(config_p))
        print('METRIC N/A')
        print('SCALAR N/A')
        print('STRICT N/A')
        metric_p = 'NA'
        scalar_p = 'NA'
        strict_p = 'NA'
    
    return(flag_passed_metric, config_p, metric_p, scalar_p, strict_p)

/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/HiTOP/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## CFA wrapper functions

In [11]:
def do_three_way_cfa_ablations(whichscale, whichcfa, howmanyitems):
    
    # checks all combinations of X items for Y scale,
    # reports if a 3-way invariant combination was found
    
    print(f'FOR SCALE {whichscale.upper()}:')    
    print(f'RUN ALL COMBINATIONS OF {howmanyitems} ITEMS and checks if a combination is 3-way invariant.')
    
    flag_found_inv_in_all_three = False
    successful_combinations = {}

    # these are the original scales
    mdl_strs_new = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop248 + hitop265',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'}
    
    for cln_ss, mdl in mdl_strs_new.items(): 
        if cln_ss == whichscale:  # take the scale we are interested in 
            if whichcfa == 'metric':
                doing_metric = True
                doing_scalar = False
                doing_strict = False
            elif whichcfa == 'scalar':
                doing_metric = True
                doing_scalar = True
                doing_strict = False   
            elif whichcfa == 'strict':
                doing_metric = True
                doing_scalar = True
                doing_strict = True                  
            print("\n")
            print(f"Running {cln_ss.upper()}")
            print(f"Items: {mdl}")
            
            # loop to do ablations
            temp_items = mdl_strs_new[cln_ss]
            temp_items_items = temp_items.split("=~",1)[1]
            temp_items_items_items = temp_items_items.split(" + ")
            list_of_items = temp_items_items_items

            count_success = 0
            count_comb = 1

            for com in combinations(list_of_items, howmanyitems):

                # number of possible combinations of x items of of n possible items
                num_combinations = math.comb(len(list_of_items), howmanyitems)
                print(f'\n+++ TESTING {count_comb}th COMBINATION out of {num_combinations} possible combinations of {howmanyitems} items +++')
                print(f'Items to test: {com}')

                # set flags for metric invariance genpop and enriched to FALSE, for this combination of items
                flag_metric_val_gp = False
                flag_metric_val_en = False
                flag_metric_gp_en = False 

                # +++ VALIDATION VS GENPOP +++
                print('\n -----> VALID VS GENPOP <----- ')
                flag_metric_val_gp, pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_val_genpop, \
                    mydata_temp_path = path_to_helpfile)
            
                # +++ VALIDATION VS ENRICHED +++ 
                print('\n -----> VALID VS ENRICHED <----- ')
                flag_metric_val_en, pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_val_enriched, \
                    mydata_temp_path = path_to_helpfile)

                # +++ GENPOP VS ENRICHED +++ 
                print('\n -----> GENPOP VS ENRICHED <----- ')
                flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_genpop_enriched_norecontact, \
                    mydata_temp_path = path_to_helpfile)

                if all([flag_metric_val_gp, flag_metric_val_en, flag_metric_gp_en]):
                    print(f"Combination {count_comb} passes 3-way invariance!")
                    flag_found_inv_in_all_three = True
                    count_success += 1
                    successful_combinations[com]={'val_gp':[pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp],\
                                                  'val_en':[pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en],\
                                                  'gp_en':[pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en]}
                count_comb+=1
                        
            print("\nCFA DONE")
            if flag_found_inv_in_all_three:
                print(f'\n\nFound {count_success} combination(s) of {howmanyitems} that is 3-way invariant.')
                print(successful_combinations)
            
            else:
                print(f'\nCould not find a combination of {howmanyitems} for scale {whichscale} that is 3-way invariant =(')
                print(f'Val–GP: {flag_metric_val_gp}')
                print(f'Val–EN: {flag_metric_val_en}')
                print(f'GP–EN:  {flag_metric_gp_en}\n')
                
    return(successful_combinations)

In [12]:
# Scalar invariance code also from claude
# =============================================================================
# Stepwise CFA for scalar invariance (v1).
# Takes a set of metric-invariant items and searches for a scalar-invariant
# core via stepwise item removal, following the same prereg-consistent
# pattern as the metric procedure:
#   - If a lower level fails after item removal (configural or metric):
#     handle that first (CFI-ablation for configural, loading MIs for metric)
#   - At scalar level: remove items based on permuted maximum INTERCEPT
#     modification index (mirrors the loading-MI logic from metric)
#
# Returns a DataFrame with one row per test (main or ablation candidate)
# and a parallel set of columns to the metric procedure plus scalar_*
# fields.
#
# Note: this implements item removal at scalar, matching the preregistration
# ("remove items from the scale in step-wise order"). The methodologically
# more common alternative is partial scalar invariance (free a single
# intercept rather than dropping the item), which is implemented separately
# elsewhere. Item removal is more aggressive but produces a cleaner "core
# set of scalar-invariant items" that matches what the preregistration
# asked for.
# =============================================================================

import pandas as pd
import random
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter


DEFAULT_CFI_TIE_TOLERANCE = 0.001
DEFAULT_TLI_TIE_TOLERANCE = 0.001


# -----------------------------------------------------------------------------
# Hierarchical CFA test for ONE pairwise comparison, up to scalar.
# -----------------------------------------------------------------------------
def cfa_test_scalar_with_mi(scalename, list_of_items, mydata_python, mydata_temp_path,
                             max_level='scalar'):
    """
    Parameters
    ----------
    max_level : {'configural', 'metric', 'scalar'}
        Highest invariance level to test. Lower levels are always tested
        first; the test sequence stops at the first failure or at max_level.

    Returns
    -------
    dict with keys:
        config_p, config_passed, config_passed_primary,
        config_cfi, config_tli, config_rmsea,
        metric_p, metric_passed, metric_item_mis,
        scalar_p, scalar_passed, scalar_item_mis
    metric_item_mis is the dict of LOADING MIs (op == "=~") populated only
    when metric failed. scalar_item_mis is the dict of INTERCEPT MIs
    (op == "~1") populated only when scalar failed.
    """
    random.seed(12345)
    ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
    ro.r('set.seed(12345)')

    myrstring = scalename + " =~ " + " + ".join(list_of_items)
    ro.globalenv['testformula_r'] = myrstring

    mydata_python.to_csv(mydata_temp_path)
    ro.r(f'rdata <- read.csv("{mydata_temp_path}", header = TRUE)')
    ro.globalenv['group'] = 'whichdata'

    ordered_str = 'c(' + ', '.join(f'"{i}"' for i in list_of_items) + ')'

    # ----- Configural -----
    fit_config = ro.r(
        f'cfa(testformula_r, data = rdata, group = group, '
        f'estimator = "WLSMV", parameterization = "theta", ordered = {ordered_str})')
    ro.globalenv['fit_config'] = fit_config
    out_config = semtools.permuteMeasEq(
        nPermute=num_iter, con=fit_config,
        parallelType="multicore", ncpus=cpus_to_use)
    config_p = float(extract_p(out_config))
    config_passed_primary = (config_p >= 0.05)
    config_passed_secondary, cfi, tli, rmsea = check_secondary_criteria(fit_config)
    config_passed = config_passed_primary or config_passed_secondary

    result = {
        'config_p':              config_p,
        'config_passed':         config_passed,
        'config_passed_primary': config_passed_primary,
        'config_cfi':            cfi,
        'config_tli':            tli,
        'config_rmsea':          rmsea,
        'metric_p':              None,
        'metric_passed':         None,
        'metric_item_mis':       None,
        'scalar_p':              None,
        'scalar_passed':         None,
        'scalar_item_mis':       None,
    }

    if max_level == 'configural' or not config_passed:
        return result

    # ----- Metric -----
    fit_metric = ro.r(
        f'cfa(testformula_r, data = rdata, group = group, '
        f'estimator = "WLSMV", parameterization = "theta", group.equal = "loadings", '
        f'ordered = {ordered_str})')
    ro.globalenv['fit_metric'] = fit_metric

    try:
        out_metric = semtools.permuteMeasEq(
            nPermute=num_iter, uncon=fit_config, con=fit_metric,
            param="loadings",
            parallelType="multicore", ncpus=cpus_to_use)
    except RRuntimeError:
        try:
            out_metric = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_config, con=fit_metric,
                param="loadings",
                parallelType="multicore", ncpus=cpus_to_use // 2)
        except RRuntimeError:
            out_metric = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_config, con=fit_metric,
                param="loadings",
                parallelType="no", ncpus=cpus_to_use // 2)

    ro.globalenv['out_metric'] = out_metric
    metric_p = float(extract_p(out_metric))
    metric_passed = (metric_p >= 0.05)
    result['metric_p'] = metric_p
    result['metric_passed'] = metric_passed
    if not metric_passed:
        result['metric_item_mis'] = extract_item_mis_from_metric(list_of_items)

    if max_level == 'metric' or not metric_passed:
        return result

    # ----- Scalar -----
    fit_scalar = ro.r(
        f'cfa(testformula_r, data = rdata, group = group, '
        f'estimator = "WLSMV", parameterization = "theta",'
        f'group.equal = c("loadings", "intercepts"), '
        f'ordered = {ordered_str})')
    ro.globalenv['fit_scalar'] = fit_scalar

    try:
        out_scalar = semtools.permuteMeasEq(
            nPermute=num_iter, uncon=fit_metric, con=fit_scalar,
            param=ro.StrVector(["loadings", "intercepts"]),
            parallelType="multicore", ncpus=cpus_to_use)
    except RRuntimeError:
        try:
            out_scalar = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_metric, con=fit_scalar,
                param=ro.StrVector(["loadings", "intercepts"]),
                parallelType="multicore", ncpus=cpus_to_use // 2)
        except RRuntimeError:
            out_scalar = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_metric, con=fit_scalar,
                param=ro.StrVector(["loadings", "intercepts"]),
                parallelType="no", ncpus=cpus_to_use // 2)

    ro.globalenv['out_scalar'] = out_scalar
    scalar_p = float(extract_p(out_scalar))
    scalar_passed = (scalar_p >= 0.05)
    result['scalar_p'] = scalar_p
    result['scalar_passed'] = scalar_passed
    if not scalar_passed:
        result['scalar_item_mis'] = extract_item_mis_from_scalar(list_of_items)

    return result


# -----------------------------------------------------------------------------
# Extract per-item INTERCEPT MIs from the scalar model's permuteMeasEq.
# At scalar, the new constraints added on top of metric are intercept
# equalities (op == "~1" in lavaan). The MIs of interest are these
# intercept MIs -- which item's intercept most differs across groups.
# -----------------------------------------------------------------------------
def extract_item_mis_from_scalar(item_list):
    """
    The scalar @MI.obs contains MIs for both loading and intercept
    equality constraints (because permuteMeasEq was called with
    param=c("loadings", "intercepts")). We filter to intercepts only --
    those are the new constraints whose violations characterize scalar
    misfit specifically.

    Returns dict[item -> max MI across non-reference groups], or None.
    """
    try:
        ro.r('''
            mi_obs <- out_scalar@MI.obs
            pt     <- out_scalar@PT
            mi_obs$par_name <- pt$par[ match(mi_obs$lhs, pt$plabel) ]
            mi_intercepts <- mi_obs[grepl("~1$", mi_obs$par_name), ]
            if (nrow(mi_intercepts) > 0) {
                mi_intercepts$item <- sub("~1$", "", mi_intercepts$par_name)
            }
        ''')
        mi_df_r = ro.r('mi_intercepts')
        with localconverter(ro.default_converter + pandas2ri.converter):
            mi_df = ro.conversion.rpy2py(mi_df_r)

        if mi_df is None or len(mi_df) == 0:
            return None
        mi_df = mi_df[mi_df['item'].isin(item_list)]
        if len(mi_df) == 0:
            return None

        item_mis = mi_df.groupby('item')['X2'].max().to_dict()
        return {k: float(v) for k, v in item_mis.items()}

    except Exception as e:
        print(f"  [WARN] Could not extract intercept MIs from scalar model: {e}")
        return None


# -----------------------------------------------------------------------------
# Pairwise comparisons and helpers
# -----------------------------------------------------------------------------
PAIRWISE_COMPARISONS = [
    ('val_gp', 'data_val_genpop'),
    ('val_en', 'data_val_enriched'),
    ('gp_en', 'data_genpop_enriched'),
]


def _test_all_three_pairs_scalar(scalename, items, max_level='scalar'):
    results = {}
    for name, datavar in PAIRWISE_COMPARISONS:
        data = globals().get(datavar) or eval(datavar)
        results[name] = cfa_test_scalar_with_mi(
            scalename, items, data, path_to_helpfile, max_level=max_level)
    return results


def _build_history_row_scalar(iteration, phase, items, candidate_dropped,
                               three_results, action=None, removed_item=None,
                               removal_reason=None, mis_used=None,
                               cfi_tied_count=None, tli_tied_count=None):
    row = {
        'iteration':         iteration,
        'phase':             phase,
        'candidate_dropped': candidate_dropped,
        'n_items':           len(items),
        'items':             tuple(items),
    }
    cfis, tlis, rmseas = [], [], []
    all_config_passed = True
    all_metric_passed = True
    all_scalar_passed = True
    metric_evaluable = True
    scalar_evaluable = True

    for name, _ in PAIRWISE_COMPARISONS:
        r = three_results[name]
        row[f'{name}_config_p']              = r['config_p']
        row[f'{name}_config_passed']         = r['config_passed']
        row[f'{name}_config_passed_primary'] = r['config_passed_primary']
        row[f'{name}_config_cfi']            = r['config_cfi']
        row[f'{name}_config_tli']            = r['config_tli']
        row[f'{name}_config_rmsea']          = r['config_rmsea']
        row[f'{name}_metric_p']              = r['metric_p']
        row[f'{name}_metric_passed']         = r['metric_passed']
        row[f'{name}_scalar_p']              = r['scalar_p']
        row[f'{name}_scalar_passed']         = r['scalar_passed']

        cfis.append(r['config_cfi'])
        tlis.append(r['config_tli'])
        rmseas.append(r['config_rmsea'])

        if not r['config_passed']:
            all_config_passed = False
            metric_evaluable = False
            scalar_evaluable = False
        if r['metric_passed'] is None:
            metric_evaluable = False
            scalar_evaluable = False
        elif not r['metric_passed']:
            all_metric_passed = False
            scalar_evaluable = False
        if r['scalar_passed'] is None:
            scalar_evaluable = False
        elif not r['scalar_passed']:
            all_scalar_passed = False

    row['min_config_cfi']    = min(cfis) if cfis else None
    row['min_config_tli']    = min(tlis) if tlis else None
    row['max_config_rmsea']  = max(rmseas) if rmseas else None
    row['all_config_passed'] = all_config_passed
    row['all_metric_passed'] = (all_metric_passed if metric_evaluable else None)
    row['all_scalar_passed'] = (all_scalar_passed if scalar_evaluable else None)
    row['action']            = action
    row['removed_item']      = removed_item
    row['removal_reason']    = removal_reason
    row['cfi_tied_count']    = cfi_tied_count
    row['tli_tied_count']    = tli_tied_count
    row['mis_used']          = mis_used
    return row


def find_worst_item(mi_dicts_failing, marker_item, current_items):
    """Same logic as in v5; sum MIs across failing comparisons, exclude marker."""
    aggregated = {}
    for item in current_items:
        if item == marker_item:
            continue
        total = 0.0
        for mi_dict in mi_dicts_failing:
            if mi_dict and item in mi_dict:
                total += mi_dict[item]
        aggregated[item] = total

    if not aggregated or max(aggregated.values()) == 0.0:
        return None, aggregated
    worst_item = max(aggregated, key=aggregated.get)
    return worst_item, aggregated


# -----------------------------------------------------------------------------
# Top-level: stepwise removal targeting scalar invariance, starting from
# a set of metric-invariant items.
# -----------------------------------------------------------------------------
def do_three_way_cfa_stepwise_scalar(whichscale,
                                      metric_invariant_items,
                                      marker_item=None,
                                      min_items=3,
                                      max_iter=None,
                                      cfi_tie_tolerance=DEFAULT_CFI_TIE_TOLERANCE,
                                      tli_tie_tolerance=DEFAULT_TLI_TIE_TOLERANCE):
    """
    Search for a 3-way scalar invariant subset starting from a metric-
    invariant core, by iteratively removing the item with the largest
    aggregated permuted modification index. Handles regressions at lower
    levels (configural, metric) defensively in case item removal causes
    them to fail again.

    Parameters
    ----------
    whichscale : str
    metric_invariant_items : list[str]
        The metric-invariant core from the upstream procedure.
    marker_item : str or None
        Item whose loading is fixed to 1 and intercept fixed to 0 (in the
        reference group). Defaults to the first item in
        metric_invariant_items, matching the original scale's marker.
    min_items : int
    max_iter : int or None
    cfi_tie_tolerance, tli_tie_tolerance : float
        Same semantics as in the metric procedure.

    Returns
    -------
    final_items : list[str] or None
    removed_in_order : list[str]
    history : pd.DataFrame
        One row per test (main or ablation candidate). Columns include
        per-pair config/metric/scalar pass-fail and p-values, aggregate
        fit indices, removal decisions, and tied-count diagnostics.
        removal_reason takes values in
        {'cfi_max_min', 'tli_tiebreaker', 'rmsea_tiebreaker',
         'metric_mi_worst', 'scalar_mi_worst', None}.
    """
    print(f"\n{'='*60}\nSTEPWISE SCALAR: {whichscale.upper()}\n{'='*60}")

    current = list(metric_invariant_items)
    if marker_item is None:
        marker_item = current[0]
    elif marker_item not in current:
        raise ValueError(f"marker_item {marker_item!r} not in input items")

    removed = []
    history_rows = []

    if max_iter is None:
        max_iter = len(current) - min_items

    for it in range(max_iter + 1):
        print(f"\n--- Iteration {it}: {len(current)} items ---")
        print(f"Current: {current}")

        if len(current) < min_items:
            print(f"Below minimum of {min_items} items. Stopping.")
            history_rows.append({
                'iteration': it, 'phase': 'final', 'n_items': len(current),
                'items': tuple(current), 'action': 'failed_min_items',
            })
            return None, removed, pd.DataFrame(history_rows)

        # --- Test current set up to scalar ---
        print("Testing current item set up to scalar...")
        main_results = _test_all_three_pairs_scalar(whichscale, current,
                                                    max_level='scalar')

        all_config = all(main_results[n]['config_passed']
                         for n, _ in PAIRWISE_COMPARISONS)
        all_metric = all(main_results[n]['metric_passed'] is True
                         for n, _ in PAIRWISE_COMPARISONS) if all_config else False
        all_scalar = all(main_results[n]['scalar_passed'] is True
                         for n, _ in PAIRWISE_COMPARISONS) if all_metric else False

        # --- Success ---
        if all_config and all_metric and all_scalar:
            print(f"\n*** 3-way scalar invariance achieved with {len(current)} items ***")
            row = _build_history_row_scalar(it, 'main', current, None,
                                            main_results, action='success')
            history_rows.append(row)
            return current, removed, pd.DataFrame(history_rows)

        # --- Configural regression: CFI ablation ---
        # This shouldn't happen since we started from a metric-invariant
        # core (which already implies configural), but handle defensively
        # in case item removal at scalar causes some pair to regress.
        if not all_config:
            print("Configural regressed for at least one comparison; running "
                  "CFI-based ablation (configural_only for candidates)...")
            row = _build_history_row_scalar(it, 'main', current, None,
                                            main_results,
                                            action='configural_ablation_starting')
            history_rows.append(row)
            main_row_idx = len(history_rows) - 1

            candidates = [i for i in current if i != marker_item]
            ablation_results = []
            for cand in candidates:
                test_items = [i for i in current if i != cand]
                if len(test_items) < min_items:
                    continue
                print(f"  Trying drop of {cand} -> {len(test_items)} items")
                cand_results = _test_all_three_pairs_scalar(
                    whichscale, test_items, max_level='configural')
                cand_row = _build_history_row_scalar(
                    it, 'ablation_candidate', test_items, cand, cand_results,
                    action='ablation_candidate_tested')
                cmc = cand_row['min_config_cfi']
                cmt = cand_row['min_config_tli']
                cmr = cand_row['max_config_rmsea']
                history_rows.append(cand_row)
                print(f"    min_cfi={cmc:.4f} min_tli={cmt:.4f} "
                      f"max_rmsea={cmr:.4f}")
                if cmc is not None:
                    ablation_results.append((cand, cmc, cmt, cmr))

            if not ablation_results:
                print("  No viable ablation candidate. Stopping.")
                history_rows.append({
                    'iteration': it, 'phase': 'final',
                    'n_items': len(current), 'items': tuple(current),
                    'action': 'failed_no_ablation_candidate',
                })
                return None, removed, pd.DataFrame(history_rows)

            # CFI -> TLI -> RMSEA tiebreaker cascade (same as in metric procedure)
            max_cfi = max(r[1] for r in ablation_results)
            cfi_tied = [r for r in ablation_results
                        if r[1] >= max_cfi - cfi_tie_tolerance]
            cfi_tied_count = len(cfi_tied)

            if cfi_tied_count == 1:
                best = cfi_tied[0]; removal_reason = 'cfi_max_min'
                tli_tied_count = None
            else:
                max_tli = max(r[2] for r in cfi_tied)
                tli_tied = [r for r in cfi_tied
                            if r[2] >= max_tli - tli_tie_tolerance]
                tli_tied_count = len(tli_tied)
                if tli_tied_count == 1:
                    best = tli_tied[0]; removal_reason = 'tli_tiebreaker'
                else:
                    best = min(tli_tied, key=lambda r: r[3])
                    removal_reason = 'rmsea_tiebreaker'

            best_candidate = best[0]
            print(f"  -> Removing {best_candidate} (reason={removal_reason})")
            history_rows[main_row_idx]['removed_item']    = best_candidate
            history_rows[main_row_idx]['removal_reason']  = removal_reason
            history_rows[main_row_idx]['cfi_tied_count']  = cfi_tied_count
            history_rows[main_row_idx]['tli_tied_count']  = tli_tied_count
            removed.append(best_candidate)
            current.remove(best_candidate)
            continue

        # --- Metric regression: loading MI removal ---
        # Again shouldn't normally happen if we started metric-invariant,
        # but defensively handle.
        if not all_metric:
            print("Metric regressed; running loading-MI-based removal...")
            failing_mis = []
            for name, _ in PAIRWISE_COMPARISONS:
                r = main_results[name]
                if r['metric_passed'] is False and r['metric_item_mis']:
                    failing_mis.append(r['metric_item_mis'])

            worst, aggregated = find_worst_item(failing_mis, marker_item, current)
            if worst is None:
                print("  Could not identify worst loading-MI item. Stopping.")
                row = _build_history_row_scalar(it, 'main', current, None,
                                                main_results,
                                                action='failed_no_metric_mi',
                                                mis_used=aggregated)
                history_rows.append(row)
                return None, removed, pd.DataFrame(history_rows)

            print(f"  Aggregated loading MIs (summed over failing pairs):")
            for item, mi in sorted(aggregated.items(), key=lambda kv: -kv[1]):
                print(f"    {item}: {mi:.3f}")
            print(f"  -> Removing: {worst}")

            row = _build_history_row_scalar(it, 'main', current, None,
                                            main_results,
                                            action='metric_mi_removal',
                                            removed_item=worst,
                                            removal_reason='metric_mi_worst',
                                            mis_used=aggregated)
            history_rows.append(row)
            removed.append(worst)
            current.remove(worst)
            continue

        # --- Scalar failure: intercept MI removal (the main case) ---
        print("Scalar failed; running intercept-MI-based removal...")
        failing_mis = []
        for name, _ in PAIRWISE_COMPARISONS:
            r = main_results[name]
            if r['scalar_passed'] is False and r['scalar_item_mis']:
                failing_mis.append(r['scalar_item_mis'])

        worst, aggregated = find_worst_item(failing_mis, marker_item, current)
        if worst is None:
            print("  Could not identify worst intercept-MI item. Stopping.")
            row = _build_history_row_scalar(it, 'main', current, None,
                                            main_results,
                                            action='failed_no_scalar_mi',
                                            mis_used=aggregated)
            history_rows.append(row)
            return None, removed, pd.DataFrame(history_rows)

        print(f"  Aggregated intercept MIs (summed over failing pairs):")
        for item, mi in sorted(aggregated.items(), key=lambda kv: -kv[1]):
            print(f"    {item}: {mi:.3f}")
        print(f"  -> Removing: {worst}")

        row = _build_history_row_scalar(it, 'main', current, None, main_results,
                                        action='scalar_mi_removal',
                                        removed_item=worst,
                                        removal_reason='scalar_mi_worst',
                                        mis_used=aggregated)
        history_rows.append(row)
        removed.append(worst)
        current.remove(worst)

    print(f"\nHit max_iter ({max_iter}) without convergence.")
    history_rows.append({
        'iteration': max_iter + 1, 'phase': 'final',
        'n_items': len(current), 'items': tuple(current),
        'action': 'failed_max_iter',
    })
    return None, removed, pd.DataFrame(history_rows)

In [13]:
# =============================================================================
# Stepwise CFA with two-tier removal (v7):
#   - Marker is now ablatable; when removed, the new first item in the
#     remaining list implicitly becomes the marker for subsequent fits.
#   - Configural failure -> bounded combinatorial ablation. Find smallest k
#     such that some k-item ablation achieves 3-way configural invariance.
#     Within that k, pick by CFI -> TLI -> RMSEA cascade. If no combo at
#     any k <= n_items - min_items passes, stop and report failure.
#   - Metric failure with configural OK -> permuted-MI removal (one item).
#     The current marker (first item) is still excluded here since its
#     loading is fixed and has no MI; this exclusion is purely local to
#     the model being fit, not a fixed assignment.
# =============================================================================

import pandas as pd
import random
from itertools import combinations
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter


DEFAULT_CFI_TIE_TOLERANCE = 0.001
DEFAULT_TLI_TIE_TOLERANCE = 0.001


def cfa_test_metric_with_mi(scalename, list_of_items, mydata_python, mydata_temp_path,
                            configural_only=False):
    random.seed(12345)
    ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
    ro.r('set.seed(12345)')

    myrstring = scalename + " =~ " + " + ".join(list_of_items)
    ro.globalenv['testformula_r'] = myrstring

    mydata_python.to_csv(mydata_temp_path)
    ro.r(f'rdata <- read.csv("{mydata_temp_path}", header = TRUE)')
    ro.globalenv['group'] = 'whichdata'

    ordered_str = 'c(' + ', '.join(f'"{i}"' for i in list_of_items) + ')'

    fit_config = ro.r(
        f'cfa(testformula_r, data = rdata, group = group, '
        f'estimator = "WLSMV", parameterization = "theta", ordered = {ordered_str})')
    ro.globalenv['fit_config'] = fit_config
    out_config = semtools.permuteMeasEq(
        nPermute=num_iter, con=fit_config,
        parallelType="multicore", ncpus=cpus_to_use)
    config_p = float(extract_p(out_config))
    config_passed_primary = (config_p >= 0.05)
    config_passed_secondary, cfi, tli, rmsea = check_secondary_criteria(fit_config)
    config_passed = config_passed_primary or config_passed_secondary

    result = {
        'config_p':              config_p,
        'config_passed':         config_passed,
        'config_passed_primary': config_passed_primary,
        'config_cfi':            cfi,
        'config_tli':            tli,
        'config_rmsea':          rmsea,
        'metric_p':              None,
        'metric_passed':         None,
        'item_mis':              None,
    }
    if configural_only or not config_passed:
        return result

    fit_metric = ro.r(
        f'cfa(testformula_r, data = rdata, group = group, '
        f'estimator = "WLSMV", parameterization = "theta", group.equal = "loadings", '
        f'ordered = {ordered_str})')
    ro.globalenv['fit_metric'] = fit_metric

    try:
        out_metric = semtools.permuteMeasEq(
            nPermute=num_iter, uncon=fit_config, con=fit_metric,
            param="loadings",
            parallelType="multicore", ncpus=cpus_to_use)
    except RRuntimeError:
        try:
            out_metric = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_config, con=fit_metric,
                param="loadings",
                parallelType="multicore", ncpus=cpus_to_use // 2)
        except RRuntimeError:
            out_metric = semtools.permuteMeasEq(
                nPermute=num_iter, uncon=fit_config, con=fit_metric,
                param="loadings",
                parallelType="no", ncpus=cpus_to_use // 2)

    ro.globalenv['out_metric'] = out_metric
    metric_p = float(extract_p(out_metric))
    metric_passed = (metric_p >= 0.05)
    result['metric_p'] = metric_p
    result['metric_passed'] = metric_passed
    if not metric_passed:
        result['item_mis'] = extract_item_mis_from_metric(list_of_items)
    return result


def extract_item_mis_from_metric(item_list):
    try:
        ro.r('''
            mi_obs <- out_metric@MI.obs
            pt     <- out_metric@PT
            mi_obs$par_name <- pt$par[ match(mi_obs$lhs, pt$plabel) ]
            mi_loadings <- mi_obs[grepl("=~", mi_obs$par_name), ]
            if (nrow(mi_loadings) > 0) {
                mi_loadings$item <- sub(".*=~", "", mi_loadings$par_name)
            }
        ''')
        mi_df_r = ro.r('mi_loadings')
        with localconverter(ro.default_converter + pandas2ri.converter):
            mi_df = ro.conversion.rpy2py(mi_df_r)

        if mi_df is None or len(mi_df) == 0:
            return None
        mi_df = mi_df[mi_df['item'].isin(item_list)]
        if len(mi_df) == 0:
            return None
        item_mis = mi_df.groupby('item')['X2'].max().to_dict()
        return {k: float(v) for k, v in item_mis.items()}
    except Exception as e:
        print(f"  [WARN] Could not extract MIs from metric model: {e}")
        return None


def find_worst_item(mi_dicts_failing, excluded_item, current_items):
    """
    Aggregate MIs across failing comparisons and return the item with the
    highest sum. excluded_item is dropped from candidacy (typically the
    current model's marker, whose loading is fixed and has no MI).
    """
    aggregated = {}
    for item in current_items:
        if item == excluded_item:
            continue
        total = 0.0
        for mi_dict in mi_dicts_failing:
            if mi_dict and item in mi_dict:
                total += mi_dict[item]
        aggregated[item] = total
    if not aggregated or max(aggregated.values()) == 0.0:
        return None, aggregated
    worst_item = max(aggregated, key=aggregated.get)
    return worst_item, aggregated


PAIRWISE_COMPARISONS = [
    ('val_gp', 'data_val_genpop'),
    ('val_en', 'data_val_enriched'),
    ('gp_en', 'data_genpop_enriched'),
]


def _test_all_three_pairs(scalename, items, configural_only=False):
    results = {}
    for name, datavar in PAIRWISE_COMPARISONS:
        data = globals().get(datavar)
        if isinstance(data, str):
            data = eval(datavar)
        results[name] = cfa_test_metric_with_mi(
            scalename, items, data, path_to_helpfile,
            configural_only=configural_only)
    return results


def _build_history_row(iteration, phase, items, candidate_dropped,
                       three_results, action=None, removed_items=None,
                       removal_reason=None, mis_used=None,
                       cfi_tied_count=None, tli_tied_count=None,
                       ablation_level=None):
    row = {
        'iteration':         iteration,
        'phase':             phase,
        'candidate_dropped': candidate_dropped,
        'n_items':           len(items),
        'items':             tuple(items),
        'ablation_level':    ablation_level,
    }
    cfis, tlis, rmseas = [], [], []
    all_config_passed = True
    all_metric_passed = True
    metric_evaluable = True

    for name, _ in PAIRWISE_COMPARISONS:
        r = three_results[name]
        row[f'{name}_config_p']              = r['config_p']
        row[f'{name}_config_passed']         = r['config_passed']
        row[f'{name}_config_passed_primary'] = r['config_passed_primary']
        row[f'{name}_config_cfi']            = r['config_cfi']
        row[f'{name}_config_tli']            = r['config_tli']
        row[f'{name}_config_rmsea']          = r['config_rmsea']
        row[f'{name}_metric_p']              = r['metric_p']
        row[f'{name}_metric_passed']         = r['metric_passed']

        cfis.append(r['config_cfi'])
        tlis.append(r['config_tli'])
        rmseas.append(r['config_rmsea'])
        if not r['config_passed']:
            all_config_passed = False
            metric_evaluable = False
        if r['metric_passed'] is None:
            metric_evaluable = False
        elif not r['metric_passed']:
            all_metric_passed = False

    row['min_config_cfi']    = min(cfis) if cfis else None
    row['min_config_tli']    = min(tlis) if tlis else None
    row['max_config_rmsea']  = max(rmseas) if rmseas else None
    row['all_config_passed'] = all_config_passed
    row['all_metric_passed'] = (all_metric_passed if metric_evaluable else None)
    row['action']            = action
    row['removed_items']     = removed_items
    row['removal_reason']    = removal_reason
    row['cfi_tied_count']    = cfi_tied_count
    row['tli_tied_count']    = tli_tied_count
    row['mis_used']          = mis_used
    return row


def _pick_by_cascade(passing_entries, cfi_tie_tolerance, tli_tie_tolerance):
    """CFI -> TLI -> RMSEA. Entries: (combo, min_cfi, min_tli, max_rmsea, all_passed)."""
    max_cfi = max(r[1] for r in passing_entries)
    cfi_tied = [r for r in passing_entries if r[1] >= max_cfi - cfi_tie_tolerance]
    cfi_tied_count = len(cfi_tied)

    if cfi_tied_count == 1:
        return cfi_tied[0], 'cfi_max_min', cfi_tied_count, None

    max_tli = max(r[2] for r in cfi_tied)
    tli_tied = [r for r in cfi_tied if r[2] >= max_tli - tli_tie_tolerance]
    tli_tied_count = len(tli_tied)

    if tli_tied_count == 1:
        return tli_tied[0], 'tli_tiebreaker', cfi_tied_count, tli_tied_count

    best = min(tli_tied, key=lambda r: r[3])
    return best, 'rmsea_tiebreaker', cfi_tied_count, tli_tied_count


def _search_combo_ablation(whichscale, current, min_items, iteration,
                            cfi_tie_tolerance, tli_tie_tolerance, history_rows):
    """
    Search the smallest combo size k such that some k-item ablation
    achieves 3-way configural invariance. ALL items in `current` are
    candidates -- the marker is now ablatable; when removed, the new
    first item implicitly becomes the marker for the resulting model.
    """
    candidates_pool = list(current)  # everything is fair game
    max_level = len(current) - min_items
    if max_level < 1:
        return None, None, None, None, None

    for level in range(1, max_level + 1):
        n_combos = sum(1 for _ in combinations(candidates_pool, level))
        print(f"  -- Ablation level {level} ({n_combos} combos) --")

        passing = []
        n_evaluated = 0
        for combo in combinations(candidates_pool, level):
            test_items = [i for i in current if i not in combo]

            combo_label = combo if len(combo) > 1 else combo[0]
            print(f"  Trying drop of {combo_label} -> {len(test_items)} items")
            cand_results = _test_all_three_pairs(whichscale, test_items,
                                                 configural_only=True)
            cand_row = _build_history_row(
                iteration, 'ablation_candidate', test_items, combo_label,
                cand_results, action='ablation_candidate_tested',
                ablation_level=level)
            cmc = cand_row['min_config_cfi']
            cmt = cand_row['min_config_tli']
            cmr = cand_row['max_config_rmsea']
            acp = cand_row['all_config_passed']
            history_rows.append(cand_row)
            print(f"    min_cfi={cmc:.4f} min_tli={cmt:.4f} "
                  f"max_rmsea={cmr:.4f} all_config_passed={acp}")

            n_evaluated += 1
            if acp:
                passing.append((combo, cmc, cmt, cmr, acp))

        if passing:
            print(f"  Level {level}: {len(passing)} of {n_evaluated} combos "
                  f"achieve 3-way configural invariance.")
            best, reason, cfi_tc, tli_tc = _pick_by_cascade(
                passing, cfi_tie_tolerance, tli_tie_tolerance)
            return best[0], reason, level, cfi_tc, tli_tc

        print(f"  Level {level}: no combo achieves 3-way configural invariance. "
              f"Expanding to level {level+1}...")

    return None, None, None, None, None


def do_three_way_cfa_stepwise_mi(whichscale,
                                  min_items=3,
                                  max_iter=None,
                                  drop_idx=None,
                                  cfi_tie_tolerance=DEFAULT_CFI_TIE_TOLERANCE,
                                  tli_tie_tolerance=DEFAULT_TLI_TIE_TOLERANCE):
    """
    Stepwise CFA with multi-level combinatorial ablation for configural
    failures and MI-based single-item removal for metric failures.

    The marker is ablatable: any item (including the first in the original
    scale) can be removed during configural ablation. After a marker
    removal, the new first item of the remaining list becomes the marker
    for subsequent model fits (lavaan's default behavior). At metric-MI
    steps, the *current* marker (current[0] at that step) is excluded
    from MI consideration because its loading is fixed and has no MI.

    Returns
    -------
    final_items : list[str] or None
    removed_in_order : list[str]
    history : pd.DataFrame
        Columns include `ablation_level` (int, when from ablation),
        `removed_items` (tuple of strings), `removal_reason` in
        {'cfi_max_min', 'tli_tiebreaker', 'rmsea_tiebreaker', 'mi_worst',
         None}.
    """
    print(f"\n{'='*60}\nSTEPWISE CFA: {whichscale.upper()}\n{'='*60}")

    items_str   = orig_items[whichscale]
    items_only  = items_str.split("=~", 1)[1].strip()
    current     = [s.strip() for s in items_only.split("+")]

    removed = []
    history_rows = []

    if drop_idx is not None:
        removed_item = current.pop(drop_idx)
        removed.append(removed_item)
        print(f"Pre-dropping item at index {drop_idx}: {removed_item}")

    if max_iter is None:
        max_iter = len(current) - min_items

    for it in range(max_iter + 1):
        print(f"\n--- Iteration {it}: {len(current)} items "
              f"(marker = {current[0]}) ---")
        print(f"Current: {current}")

        if len(current) < min_items:
            print(f"Below minimum of {min_items} items. Stopping.")
            history_rows.append({
                'iteration': it, 'phase': 'final', 'n_items': len(current),
                'items': tuple(current), 'action': 'failed_min_items',
            })
            return None, removed, pd.DataFrame(history_rows)

        print("Testing current item set...")
        main_results = _test_all_three_pairs(whichscale, current,
                                             configural_only=False)
        all_config = all(main_results[n]['config_passed']
                         for n, _ in PAIRWISE_COMPARISONS)
        all_metric = all(main_results[n]['metric_passed'] is True
                         for n, _ in PAIRWISE_COMPARISONS) if all_config else False

        if all_config and all_metric:
            print(f"\n*** 3-way metric invariance achieved with {len(current)} items ***")
            row = _build_history_row(it, 'main', current, None, main_results,
                                     action='success')
            history_rows.append(row)
            return current, removed, pd.DataFrame(history_rows)

        # ----- Configural failure: combinatorial ablation (marker ablatable) -----
        if not all_config:
            print("Configural failed for at least one comparison; running "
                  "combinatorial ablation search (all items, incl. marker)...")
            row = _build_history_row(it, 'main', current, None, main_results,
                                     action='configural_ablation_starting')
            history_rows.append(row)
            main_row_idx = len(history_rows) - 1

            best_combo, reason, level, cfi_tc, tli_tc = _search_combo_ablation(
                whichscale, current, min_items, it,
                cfi_tie_tolerance, tli_tie_tolerance, history_rows)

            if best_combo is None:
                print("  No combo at any level achieves 3-way configural "
                      "invariance. Stopping.")
                history_rows[main_row_idx]['action'] = 'failed_no_combo_passes'
                history_rows.append({
                    'iteration': it, 'phase': 'final',
                    'n_items': len(current), 'items': tuple(current),
                    'action': 'failed_no_combo_passes',
                })
                return None, removed, pd.DataFrame(history_rows)

            print(f"  -> Removing combo {best_combo} "
                  f"(ablation_level={level}, reason={reason})")
            history_rows[main_row_idx]['removed_items']    = tuple(best_combo)
            history_rows[main_row_idx]['removal_reason']   = reason
            history_rows[main_row_idx]['ablation_level']   = level
            history_rows[main_row_idx]['cfi_tied_count']   = cfi_tc
            history_rows[main_row_idx]['tli_tied_count']   = tli_tc

            for item in best_combo:
                removed.append(item)
                current.remove(item)
            continue

        # ----- Configural OK, metric failed: single-item MI removal -----
        # The marker for the *current* fit is current[0]; exclude it from
        # MI candidacy (its loading is fixed). If a prior ablation step
        # changed who current[0] is, that change naturally carries through.
        print(f"Configural OK but metric failed; running MI-based removal "
              f"(excluding current marker {current[0]})...")
        failing_mis = []
        for name, _ in PAIRWISE_COMPARISONS:
            r = main_results[name]
            if r['metric_passed'] is False and r['item_mis']:
                failing_mis.append(r['item_mis'])

        worst, aggregated = find_worst_item(failing_mis,
                                            excluded_item=current[0],
                                            current_items=current)

        if worst is None:
            print("  Could not identify a worst item from MIs. Stopping.")
            row = _build_history_row(it, 'main', current, None, main_results,
                                     action='failed_no_mi', mis_used=aggregated)
            history_rows.append(row)
            return None, removed, pd.DataFrame(history_rows)

        print(f"  Aggregated MIs (summed over failing comparisons):")
        for item, mi in sorted(aggregated.items(), key=lambda kv: -kv[1]):
            print(f"    {item}: {mi:.3f}")
        print(f"  -> Removing: {worst}")

        row = _build_history_row(it, 'main', current, None, main_results,
                                 action='metric_mi_removal',
                                 removed_items=(worst,),
                                 removal_reason='mi_worst',
                                 mis_used=aggregated)
        history_rows.append(row)

        removed.append(worst)
        current.remove(worst)

    print(f"\nHit max_iter ({max_iter}) without convergence.")
    history_rows.append({
        'iteration': max_iter + 1, 'phase': 'final',
        'n_items': len(current), 'items': tuple(current),
        'action': 'failed_max_iter',
    })
    return None, removed, pd.DataFrame(history_rows)

In [14]:
def run_specific_cfa(whichscale, item_list, whichcfa, return_vals=False):
    # checks 3-way CFA for a specific formula
    print(f'\n\nFOR SCALE {whichscale.upper()}:')    
    print(f'\nRUN ALL 3-way CFA for items: {item_list}.')
    
    # down to which scale to test
    if whichcfa == 'metric':
        doing_metric = True
        doing_scalar = False
        doing_strict = False
    elif whichcfa == 'scalar':
        doing_metric = True
        doing_scalar = True
        doing_strict = False   
    elif whichcfa == 'strict':
        doing_metric = True
        doing_scalar = True
        doing_strict = True  

    # set flags for metric invariance to FALSE, for this combination of items
    flag_metric_val_gp = False
    flag_metric_val_en = False
    flag_metric_gp_en = False 

    res = []
    # +++ VALIDATION VS GENPOP +++
    print('\n -----> VALID VS GENPOP <----- ')
    flag_metric_val_gp, pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_val_genpop, \
                    mydata_temp_path = path_to_helpfile)
    row = dict(
        pair='V_GP',
        pconfig=pconfig_val_gp,
        pmetric=pmetric_val_gp,
        pscalar=pscalar_val_gp,
        pstrict=pstrict_val_gp
    )
    res.append(row)
    # +++ VALIDATION VS ENRICHED +++ 
    print('\n -----> VALID VS ENRICHED <----- ')
    flag_metric_val_en, pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_val_enriched, \
                    mydata_temp_path = path_to_helpfile)
    row = dict(
        pair='V_EN',
        pconfig=pconfig_val_en,
        pmetric=pmetric_val_en,
        pscalar=pscalar_val_en,
        pstrict=pstrict_val_en
    )
    res.append(row)
    # +++ GENPOP VS ENRICHED +++ 
    print('\n -----> GENPOP VS ENRICHED <----- ')
    flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_genpop_enriched, \
                    mydata_temp_path = path_to_helpfile)
    row = dict(
        pair='GP_EN',
        pconfig=pconfig_gp_en,
        pmetric=pmetric_gp_en,
        pscalar=pscalar_gp_en,
        pstrict=pstrict_gp_en
    )
    res.append(row)

    if all([flag_metric_val_gp, flag_metric_val_en, flag_metric_gp_en]):
        flag_found_inv_in_all_three = True
        print("\n!!! PASSES 3-WAY INVARIANCE")
                   
    print("DONE")
    if return_vals:
        return res
    else: 
        return(0)

# Run Main Code

## Load preprocessed data and concatinate

In [15]:
# load
data_val = pd.read_csv(path_save_val)

data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])

data_gp_nore = pd.read_csv(path_save_dat_gp_grid1st_norecontact)
data_en_nore = pd.read_csv(path_save_dat_en_grid1st_norecontact)
data_genpop_enriched_norecontact = pd.concat([data_gp_nore, data_en_nore])

# Find invariant subsets via stepwise elimination

In [23]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'indecisiveness': 'indecisiveness =~hitop21 + hitop90 + hitop95',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}
            

In [24]:
with open("log/mylog_3wayCFA_origscales_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        orig_icc_res = []
        for scale, items in orig_items.items():
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            icc_res = run_specific_cfa(
                whichscale=scale,
                item_list=items_list, \
                whichcfa='strict',
                return_vals=True
            )
            icc_res = pd.DataFrame(icc_res)
            icc_res['scale'] = scale
            orig_icc_res.append(icc_res)
orig_icc_res = pd.concat(orig_icc_res)


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

In [25]:
orig_icc_res = orig_icc_res.astype(float, errors='ignore')
orig_icc_res = orig_icc_res.replace("NA", pd.NA)

In [26]:
orig_icc_res.to_csv(log_dir / 'orig_icc_res.csv', index=None)

for a set of items:  
    if it's 3-way config and 3-way metric:  
        return set of items and list of removed items  
    if it's not 3-way config invariant:  
        evaluate configural invariance on all single item ablations
        pick the 3-way config invariant ablation that has the best CFI (with TLI tibreaker if the dif in CFI is less than 0.001 across all 3 pairs  
        restart the loop with the selected set of items  
    if it's 3-way config invariant, but not 3-way metric:  
       remove the item with the highest modification index
       restart the loop with the selected set of items

In [27]:
# silence R to clean up messages
# be careful doing this, you might miss important warnings
import rpy2.rinterface_lib.callbacks

# Define a function that does nothing
def silence_r(x):
    pass

# Override both standard output and warning/error output
rpy2.rinterface_lib.callbacks.consolewrite_print = silence_r
rpy2.rinterface_lib.callbacks.consolewrite_warnerror = silence_r

In [28]:
stepwise_res = []
histories = []
scales_failing_metric = [
    'anhedonic_depression',
    'anxious_worry',
    'appetite_gain',
    'hyposomnia',
    'panic',
    'separation_insecurity',
    'situational_phobia',
    'shame_guilt',
    'social_anxiety',
    'well_being',
]
for scale in scales_failing_metric:
    final, removed, history = do_three_way_cfa_stepwise_mi(
        scale,
        min_items=3,
    )
    if final is not None:
        final_items = [item_lut[item_no] for item_no in final]
        removed_items = [item_lut[item_no] for item_no in removed]
        row = dict(
            scale=scale,
            item_nos=final,
            removed_nos=removed,
            items=final_items,
            removed=removed_items
        )
    else:
        row = dict(
            scale=scale
        )
    
    stepwise_res.append(row)
    pd.DataFrame(stepwise_res).to_pickle(cfa_dir / 'stepwise_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_history.pkl')
    histories.append(history)


STEPWISE CFA: ANHEDONIC_DEPRESSION

--- Iteration 0: 10 items (marker = hitop39) ---
Current: ['hitop39', 'hitop77', 'hitop84', 'hitop92', 'hitop93', 'hitop123', 'hitop157', 'hitop182', 'hitop230', 'hitop246']
Testing current item set...
Configural failed for at least one comparison; running combinatorial ablation search (all items, incl. marker)...
  -- Ablation level 1 (10 combos) --
  Trying drop of hitop39 -> 9 items
    min_cfi=0.9160 min_tli=0.8890 max_rmsea=0.1250 all_config_passed=False
  Trying drop of hitop77 -> 9 items
    min_cfi=0.8830 min_tli=0.8450 max_rmsea=0.1470 all_config_passed=False
  Trying drop of hitop84 -> 9 items
    min_cfi=0.8690 min_tli=0.8250 max_rmsea=0.1510 all_config_passed=False
  Trying drop of hitop92 -> 9 items
    min_cfi=0.9210 min_tli=0.8950 max_rmsea=0.1150 all_config_passed=False
  Trying drop of hitop93 -> 9 items
    min_cfi=0.8990 min_tli=0.8650 max_rmsea=0.1260 all_config_passed=False
  Trying drop of hitop123 -> 9 items
    min_cfi=0.8880

In [29]:
stepwise_res = pd.DataFrame(stepwise_res)

In [30]:
stepwise_res

,scale,item_nos,removed_nos,items,removed
0,anhedonic_depression,"[hitop77, hitop84, hitop93, hitop123, hitop182, hitop230, hitop246]","[hitop39, hitop157, hitop92]","[I didn’t look forward to seeing friends or family., I felt depressed., Nothing seemed interesti...","[It felt like there wasn’t anything interesting or fun to do., I had very little energy., It too..."
1,anxious_worry,NaN,NaN,NaN,NaN
2,appetite_gain,"[hitop120, hitop243, hitop275]",[hitop141],"[I could not keep myself from eating., I stuffed myself with food., I ate even when I was not re...",[I thought a lot about food.]
3,hyposomnia,"[hitop99, hitop5, hitop66, hitop231]",[hitop181],"[I needed much less sleep than usual., I had days when I never got tired., I did not feel tired,...",[I felt like I could keep going and going without ever getting tired.]
4,panic,"[hitop15, hitop104, hitop126, hitop215, hitop257]",[hitop211],"[I was short of breath., I felt nauseated., My heart was racing or pounding., My hands were cold...",[I was trembling or shaking.]
5,separation_insecurity,"[hitop40, hitop69, hitop81, hitop113, hitop136, hitop151]","[hitop50, hitop197]","[I felt insecure about important relationships in my life., I wanted other people to take care o...","[I wanted someone else to make decisions for me., I could not stand being alone.]"
6,situational_phobia,"[hitop16, hitop165, hitop278]","[hitop225, hitop247]","[I avoided riding in elevators., I was afraid of flying., I became very anxious during a storm.]","[I was afraid of the dark., I was afraid of heights.]"
7,shame_guilt,"[hitop72, hitop140, hitop220]",[hitop143],"[I was disgusted with myself., I blamed myself for things., I felt ashamed of things I had done.]",[I felt guilty.]
8,social_anxiety,"[hitop124, hitop222, hitop258]","[hitop1, hitop117, hitop204, hitop236, hitop129, hitop114, hitop17]","[I avoided situations in which others were likely to watch me., I was uncomfortable entering a r...","[I felt shy around other people., I felt socially awkward., I felt uncomfortable being the cente..."
9,well_being,"[hitop9, hitop23, hitop149, hitop200, hitop244, hitop250, hitop281]","[hitop106, hitop54, hitop245]","[I felt like I was having a lot of fun., I felt cheerful., I felt good about myself., I found co...","[I was proud of myself., It was easy for me to laugh., I looked forward to things with enjoyment.]"


In [31]:
stepwise_res.to_pickle(cfa_dir / 'stepwise.pkl')

# Exploratory: check metric invariance of PHQ, GAD, and BAARS

In [33]:
other_scales = { 
    'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'} 

other_log = log_dir /'mylog_3wayCFA_baarsgadphq_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")
        
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
        
            # +++ GENPOP VS ENRICHED +++ 
            print('\n -----> GENPOP VS ENRICHED <----- ')
            flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                            scalename = scale,\
                            list_of_items = items_list,\
                            do_metric = True, \
                            do_scalar = True, \
                            do_strict = True, \
                            mydata_python = data_genpop_enriched, \
                            mydata_temp_path = path_to_helpfile)   

In [34]:
def do_two_way_cfa_ablations(whichscale, whichcfa, howmanyitems):
    
    # checks all combinations of X items for Y scale,
    # reports if a 3-way invariant combination was found
    
    print(f'FOR SCALE {whichscale.upper()}:')    
    print(f'RUN ALL COMBINATIONS OF {howmanyitems} ITEMS and checks if a combination is 3-way invariant.')
    
    flag_found_inv_in_all_three = False
    successful_combinations = {}

    # these are the original scales
    mdl_strs_new = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop248 + hitop265',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281',
        'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
        'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
        'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
        'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
        'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
        'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}
    
    for cln_ss, mdl in mdl_strs_new.items(): 
        if cln_ss == whichscale:  # take the scale we are interested in 
            if whichcfa == 'metric':
                doing_metric = True
                doing_scalar = False
                doing_strict = False
            elif whichcfa == 'scalar':
                doing_metric = True
                doing_scalar = True
                doing_strict = False   
            elif whichcfa == 'strict':
                doing_metric = True
                doing_scalar = True
                doing_strict = True                  
            print("\n")
            print(f"Running {cln_ss.upper()}")
            print(f"Items: {mdl}")
            
            # loop to do ablations
            temp_items = mdl_strs_new[cln_ss]
            temp_items_items = temp_items.split("=~",1)[1]
            temp_items_items_items = temp_items_items.split(" + ")
            list_of_items = temp_items_items_items

            count_success = 0
            count_comb = 1

            for com in combinations(list_of_items, howmanyitems):

                # number of possible combinations of x items of of n possible items
                num_combinations = math.comb(len(list_of_items), howmanyitems)
                print(f'\n+++ TESTING {count_comb}th COMBINATION out of {num_combinations} possible combinations of {howmanyitems} items +++')
                print(f'Items to test: {com}')

                # set flags for metric invariance genpop and enriched to FALSE, for this combination of items
                flag_metric_gp_en = False 


                # +++ GENPOP VS ENRICHED +++ 
                print('\n -----> GENPOP VS ENRICHED <----- ')
                flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_genpop_enriched, \
                    mydata_temp_path = path_to_helpfile)

                if flag_metric_gp_en:
                    print(f"Combination {count_comb} passes 3-way invariance!")
                    count_success += 1
                    successful_combinations[com]={'gp_en':[pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en]}
                count_comb+=1
                        
            print("\nCFA DONE")
            if flag_metric_gp_en:
                print(f'\n\nFound {count_success} combination(s) of {howmanyitems} that is 3-way invariant.')
                print(successful_combinations)
            
            else:
                print(f'\nCould not find a combination of {howmanyitems} for scale {whichscale} that is 3-way invariant =(')
                print(f'GP–EN:  {flag_metric_gp_en}\n')
                
    return(successful_combinations)

In [35]:
other_exhaustive_logs = log_dir /'mylog_exhuastive_inv_search_baarsgadphq_seed12345.txt'

with other_exhaustive_logs.open("w") as f:
    with redirect_stdout(f):
        for scale in ['phq_sum', 'gad_sum', 'baars_inattention_sum', 'baars_sct_sum']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = other_scales[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = do_two_way_cfa_ablations(whichscale=scale, whichcfa='metric', howmanyitems=how_many_to_try)
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale: 
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

In [36]:
import ast

In [37]:
lfi_out = other_exhaustive_logs.read_text().split('\n')
lfi_parsing = []
for lix, line in enumerate(lfi_out):
    if line.startswith('!!!!!'):
        row= dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        lfi_parsing.append(row)
ds_pairs = ['gp_en']
inv_levels = ['config', 'metric', 'scalar', 'strict']

In [38]:
inv_dat = []
for lprow, olut  in zip(lfi_parsing, [phq_lut, gad_lut, baars_lut['inattention'], baars_lut['sct']]):
    scale_stats = ast.literal_eval(lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        row = lprow.copy()
        item_nos = iss[0]
        ogitems = list(olut.keys())
        items = [olut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [olut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        inv_dat.append(row)
inv_dat = pd.DataFrame(inv_dat)

In [39]:
for row in inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for ii in row.removed:
        print(ii)
    print()
    print()

########################
Scale: phq_sum, Subset_id: 0
########################
Little interest or pleasure in doing things
Trouble falling or staying asleep, or sleeping too much
Feeling tired or having little energy
Poor appetite or overeating
Feeling bad about yourself – or that you are a failure or have let yourself or your family down
Trouble concentrating on things, such as school work, reading or watching television
---------REMOVED---------------
Feeling down, depressed, irritable or hopeless
Moving or speaking so slowly that other people could have noticed? Or the opposite – being so fidgety or restless that you have been moving around a lot more than usual


########################
Scale: gad_sum, Subset_id: 0
########################
Feeling nervous, anxious, or on edge
Not being able to stop or control worrying
Worrying too much about different things
Trouble relaxing
Being so restless that it is hard to sit still
Feeling afraid, as if something awful might happen
---------